# Paper Figures 6 And 7: ASHD Forecast Horizon Results

This notebook regenerates draft-style Figures 6 and 7 from the current processed ASHD 148-household forecast-horizon results. It does not rerun PyNNLF experiments.

## 1. Purpose, Inputs, And Outputs

Input: `results/02_ashd_148hh_forecast_horizon/ashd_148hh_horizon_combined_recap.csv`.

Figure 6 output: `results/02_ashd_148hh_forecast_horizon/figures/paper_figure_ashd_horizon_key_models_test_nrmse.png`.

Figure 7 output: `results/02_ashd_148hh_forecast_horizon/figures/paper_figure_ashd_fh10_nrmse_stddev_scatter.png`.

Figure 6 compares current best, naive, and ARIMA across the 30-minute, 1-day, and 1-week horizons. Figure 7 plots 1-week test nRMSE against test nRMSE standard deviation for all models.

## 2. Setup

In [1]:

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if candidate.name == "journal_article_1" and (candidate / "results").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "publication" / "journal_article_1"
        if (nested / "results").exists() and (nested / "notebooks").exists():
            return nested.resolve()
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


PROJECT_DIR = find_publication_project()
RESULTS_DIR = PROJECT_DIR / "results"
print(f"Publication project: {PROJECT_DIR}")

MODEL_LABELS = {
    "m1_naive_hp1": "naive_hp1",
    "m2_snaive_hp2": "snaive_hp2",
    "m3_ets_hp1": "ets_hp1",
    "m4_arima_hp1": "arima_hp1",
    "m6_lr_hp1": "lr_hp1",
    "m7_ann_hp1": "ann_hp1",
    "m8_dnn_hp1": "dnn_hp1",
    "m9_rt_hp3": "rt_hp3",
    "m10_rf_hp1": "rf_hp1",
    "m13_lstm_hp2": "lstm_hp2",
    "m16_prophet_hp1": "prophet_hp1",
    "m17_xgb_hp1": "xgb_hp1",
}
MODEL_ORDER = list(MODEL_LABELS)
KEY_MODELS = {
    "Naive": "m1_naive_hp1",
    "ARIMA": "m4_arima_hp1",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})


def short_model_name(model_name: str) -> str:
    return MODEL_LABELS.get(str(model_name), str(model_name))


def save_figure(fig, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved: {path}")

SECTION_DIR = RESULTS_DIR / "02_ashd_148hh_forecast_horizon"
FIGURES_DIR = SECTION_DIR / "figures"
INPUT_PATH = SECTION_DIR / "ashd_148hh_horizon_combined_recap.csv"
FIGURE_06_PATH = FIGURES_DIR / "paper_figure_ashd_horizon_key_models_test_nrmse.png"
FIGURE_07_PATH = FIGURES_DIR / "paper_figure_ashd_fh10_nrmse_stddev_scatter.png"
HORIZON_ORDER = ["30_min", "1_day", "1_week"]
HORIZON_LABELS = {
    "30_min": "30 min",
    "1_day": "1 day",
    "1_week": "1 week",
}
HORIZON_COLORS = {
    "30_min": "#2F4D67",
    "1_day": "#EB932C",
    "1_week": "#5D7F3F",
}
print(f"Input: {INPUT_PATH}")
print(f"Figure 6 output: {FIGURE_06_PATH}")
print(f"Figure 7 output: {FIGURE_07_PATH}")


Publication project: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1
Input: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\results\02_ashd_148hh_forecast_horizon\ashd_148hh_horizon_combined_recap.csv
Figure 6 output: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\results\02_ashd_148hh_forecast_horizon\figures\paper_figure_ashd_horizon_key_models_test_nrmse.png
Figure 7 output: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\results\02_ashd_148hh_forecast_horizon\figures\paper_figure_ashd_fh10_nrmse_stddev_scatter.png


## 3. Load And Validate Current Results

In [2]:

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing processed result file: {INPUT_PATH}")

recap = pd.read_csv(INPUT_PATH)
required_columns = {
    "horizon_label",
    "forecast_horizon_min",
    "model_name",
    "test_nRMSE",
    "test_nRMSE_stddev",
}
missing_columns = sorted(required_columns - set(recap.columns))
if missing_columns:
    raise ValueError(f"Missing required columns in {INPUT_PATH.name}: {missing_columns}")

recap = recap.loc[recap["horizon_label"].isin(HORIZON_ORDER)].copy()
for column in ["forecast_horizon_min", "test_nRMSE", "test_nRMSE_stddev"]:
    recap[column] = pd.to_numeric(recap[column], errors="coerce")
if recap.empty:
    raise ValueError("No ASHD horizon rows found in the combined recap.")
if recap[["test_nRMSE", "test_nRMSE_stddev"]].isna().any().any():
    bad = recap.loc[recap[["test_nRMSE", "test_nRMSE_stddev"]].isna().any(axis=1)]
    raise ValueError("Missing numeric nRMSE values found:\n" + bad[["horizon_label", "model_name", "test_nRMSE", "test_nRMSE_stddev"]].to_string(index=False))

for horizon in HORIZON_ORDER:
    horizon_rows = recap.loc[recap["horizon_label"].eq(horizon)]
    missing_models = sorted(set(MODEL_ORDER) - set(horizon_rows["model_name"].astype(str)))
    if missing_models:
        raise ValueError(f"{horizon} is missing expected models: {missing_models}")
    for model in KEY_MODELS.values():
        if model not in set(horizon_rows["model_name"].astype(str)):
            raise ValueError(f"{horizon} is missing key model {model}")

summary = recap.groupby("horizon_label", observed=False).agg(
    rows=("model_name", "count"),
    best_test_nRMSE=("test_nRMSE", "min"),
).reindex(HORIZON_ORDER)
display(summary)


               rows  best_test_nRMSE
horizon_label                       
30_min           12         2.611803
1_day            12         6.087214
1_week           12         6.610976


## 4. Select Current Best, Naive, And ARIMA By Horizon

In [3]:

best_by_horizon = {}
for horizon in HORIZON_ORDER:
    horizon_rows = recap.loc[recap["horizon_label"].eq(horizon)].sort_values(["test_nRMSE", "model_name"])
    best_by_horizon[horizon] = str(horizon_rows.iloc[0]["model_name"])

unique_best_models = sorted(set(best_by_horizon.values()))
if len(unique_best_models) != 1:
    raise ValueError(
        "Figure 6 uses a single y-axis label for the best model, but the current best model differs by horizon: "
        + str({HORIZON_LABELS[k]: short_model_name(v) for k, v in best_by_horizon.items()})
    )
best_model = unique_best_models[0]

plot_rows = []
row_specs = [(short_model_name(best_model), best_model), ("naive_hp1", KEY_MODELS["Naive"]), ("arima_hp1", KEY_MODELS["ARIMA"])]
for row_label, model_name in row_specs:
    for horizon in HORIZON_ORDER:
        match = recap.loc[recap["horizon_label"].eq(horizon) & recap["model_name"].astype(str).eq(model_name)]
        if match.empty:
            raise ValueError(f"Missing {horizon} / {model_name}")
        item = match.iloc[0]
        plot_rows.append({
            "row_label": row_label,
            "horizon_label": horizon,
            "horizon_display": HORIZON_LABELS[horizon],
            "model_name": model_name,
            "model_display": short_model_name(model_name),
            "test_nRMSE": float(item["test_nRMSE"]),
            "test_nRMSE_stddev": float(item["test_nRMSE_stddev"]),
        })

plot_df = pd.DataFrame(plot_rows)
display(plot_df)


   row_label horizon_label  ... test_nRMSE test_nRMSE_stddev
0    xgb_hp1        30_min  ...   2.611803          0.242532
1    xgb_hp1         1_day  ...   6.087214          0.546110
2    xgb_hp1        1_week  ...   6.610976          0.559662
3  naive_hp1        30_min  ...   4.224911          0.537976
4  naive_hp1         1_day  ...   9.091399          0.904674
5  naive_hp1        1_week  ...  10.937228          0.855400
6  arima_hp1        30_min  ...   3.557113          0.340208
7  arima_hp1         1_day  ...  16.178947          1.980822
8  arima_hp1        1_week  ...  17.847139          2.402397

[9 rows x 7 columns]


## 5. Create Figure 6

In [4]:

row_order = list(dict.fromkeys(plot_df["row_label"].tolist()))
y = np.arange(len(row_order))
bar_height = 0.24
fig, ax = plt.subplots(figsize=(8.2, 4.6))

for i, horizon in enumerate(HORIZON_ORDER):
    subset = plot_df.loc[plot_df["horizon_label"].eq(horizon)].set_index("row_label").reindex(row_order)
    offset = (i - 1) * bar_height
    ax.barh(
        y + offset,
        subset["test_nRMSE"],
        xerr=subset["test_nRMSE_stddev"],
        height=bar_height,
        label=HORIZON_LABELS[horizon],
        color=HORIZON_COLORS[horizon],
        alpha=0.92,
        capsize=3,
        error_kw={"elinewidth": 0.9, "alpha": 0.85},
    )

ax.set_yticks(y)
ax.set_yticklabels(row_order)
ax.invert_yaxis()
ax.set_xlabel("Test nRMSE (%)")
ax.set_title("ASHD 148-household dataset: forecast-horizon sensitivity")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=True)
ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", visible=False)
right_edge = (plot_df["test_nRMSE"] + plot_df["test_nRMSE_stddev"]).max()
ax.set_xlim(0, right_edge * 1.06)
fig.tight_layout()
save_figure(fig, FIGURE_06_PATH)
plt.show()


Saved: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\results\02_ashd_148hh_forecast_horizon\figures\paper_figure_ashd_horizon_key_models_test_nrmse.png
publication\journal_article_1\notebooks\02_ashd_148hh_forecast_horizon\4_create_paper_figures_06_07_ashd_horizons.ipynb:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "## 2. Setup"


## 6. Create Figure 7

In [5]:

scatter = recap.loc[recap["horizon_label"].eq("1_week")].copy()
if scatter.empty:
    raise ValueError("No 1-week ASHD rows found for Figure 7.")
if scatter["model_name"].nunique() != len(MODEL_ORDER):
    raise ValueError(f"Expected {len(MODEL_ORDER)} unique models for Figure 7, found {scatter['model_name'].nunique()}.")
scatter["model_display"] = scatter["model_name"].astype(str).map(MODEL_LABELS).fillna(scatter["model_name"].astype(str))
scatter = scatter.sort_values("test_nRMSE")

# Hand-tuned label positions keep the dense 8.5-11.5% nRMSE cluster readable.
# Leader lines are drawn after the text boxes are rendered, stopping at the box edge.
label_positions = {
    "xgb_hp1": (6.85, 0.68),
    "dnn_hp1": (8.20, 1.02),
    "lr_hp1": (8.10, 0.52),
    "ann_hp1": (8.10, 0.38),
    "rf_hp1": (8.55, 0.24),
    "rt_hp3": (8.20, 1.18),
    "prophet_hp1": (9.65, 1.82),
    "lstm_hp2": (10.35, 1.66),
    "naive_hp1": (11.45, 1.08),
    "snaive_hp2": (11.60, 0.63),
    "ets_hp1": (11.95, 1.20),
    "arima_hp1": (17.00, 2.18),
}

fig, ax = plt.subplots(figsize=(7.6, 5.6))
ax.scatter(
    scatter["test_nRMSE"],
    scatter["test_nRMSE_stddev"],
    s=58,
    color="#2F4D67",
    edgecolor="white",
    linewidth=0.8,
    zorder=3,
)

text_items = []
for row in scatter.itertuples(index=False):
    label_xy = label_positions.get(row.model_display, (row.test_nRMSE + 0.3, row.test_nRMSE_stddev + 0.05))
    text = ax.text(
        label_xy[0],
        label_xy[1],
        row.model_display,
        ha="left",
        va="center",
        fontsize=8,
        bbox={"facecolor": "white", "edgecolor": "#CCCCCC", "alpha": 0.9, "boxstyle": "round,pad=0.18"},
        zorder=5,
    )
    text_items.append((float(row.test_nRMSE), float(row.test_nRMSE_stddev), text))

ax.set_xlabel("Test nRMSE (%)")
ax.set_ylabel("Test nRMSE stddev (%)")
ax.set_title("ASHD 148-household dataset, 1-week forecast horizon")
ax.grid(True, alpha=0.25)
ax.set_xlim(5.7, 18.6)
ax.set_ylim(0.15, 2.62)

# Draw leader lines to the outside edge of each rendered label box, with a small gap.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
inv = ax.transData.inverted()
for point_x, point_y, text in text_items:
    bbox = text.get_window_extent(renderer=renderer).expanded(1.08, 1.18)
    (x0, y0), (x1, y1) = inv.transform([[bbox.x0, bbox.y0], [bbox.x1, bbox.y1]])
    center_x = (x0 + x1) / 2
    center_y = (y0 + y1) / 2
    vx = point_x - center_x
    vy = point_y - center_y
    candidates = []
    if abs(vx) > 1e-12:
        candidates.append(((x1 - center_x) / vx) if vx > 0 else ((x0 - center_x) / vx))
    if abs(vy) > 1e-12:
        candidates.append(((y1 - center_y) / vy) if vy > 0 else ((y0 - center_y) / vy))
    positive_candidates = [value for value in candidates if value > 0]
    if not positive_candidates:
        continue
    scale = min(positive_candidates)
    end_x = center_x + vx * scale
    end_y = center_y + vy * scale
    ax.plot(
        [point_x, end_x],
        [point_y, end_y],
        color="#777777",
        linewidth=0.7,
        zorder=2,
        solid_capstyle="round",
    )

fig.tight_layout()
save_figure(fig, FIGURE_07_PATH)
plt.show()

display(scatter[["model_display", "test_nRMSE", "test_nRMSE_stddev"]])


Saved: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\results\02_ashd_148hh_forecast_horizon\figures\paper_figure_ashd_fh10_nrmse_stddev_scatter.png
publication\journal_article_1\notebooks\02_ashd_148hh_forecast_horizon\4_create_paper_figures_06_07_ashd_horizons.ipynb:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "\n",
   model_display  test_nRMSE  test_nRMSE_stddev
35       xgb_hp1    6.610976           0.559662
30       dnn_hp1    8.577552           0.844146
28        lr_hp1    8.756670           0.744845
29       ann_hp1    8.835541           0.789687
32        rf_hp1    8.994642           0.652715
34   prophet_hp1    9.740858           1.395916
33      lstm_hp2   10.113575           1.477309
31        rt_hp3   10.444945           0.903832
24     naive_hp1   10.937228           0.855400
25    snaive_hp2   10.937228           0.855400
26       ets_hp1   10.980918           0.895377
27     arima_hp1   17.847139           2.402397


## 7. Saved File Summary

In [6]:

for path in [FIGURE_06_PATH, FIGURE_07_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Expected figure was not created: {path}")
    print(f"{path.name}: {path.stat().st_size:,} bytes")


paper_figure_ashd_horizon_key_models_test_nrmse.png: 77,152 bytes
paper_figure_ashd_fh10_nrmse_stddev_scatter.png: 141,123 bytes
